# 01 — Chất lượng và chuẩn bị dữ liệu

**Phụ trách:** Tiết Trọng Khiêm

**Trạng thái:** Đã triển khai các bước kiểm định chất lượng dữ liệu, chuẩn bị tập core và phân tích độ bao phủ của `abuse_type`.

**Đầu ra:** Bảng kiểm kê và chất lượng dữ liệu, tập core `gold` + `silver`, tập `bronze` được giữ riêng, cùng các nhận xét về quy tắc lọc và hạn chế của dữ liệu.

## 0. Môi trường và đường dẫn
Mặc định chạy local: cài requirements.txt trong .venv, mở notebook bằng VS Code/Jupyter và chọn kernel của .venv. Dataset ở data/ hoặc vị trí PHISHGUARD_DATA_DIR trên máy mình. Ô dưới chỉ nạp đường dẫn/config; Colab là lựa chọn thêm theo README.

In [36]:
from pathlib import Path
import os
import sys
import tomllib

start = Path(os.environ.get("PHISHGUARD_ROOT", Path.cwd())).expanduser().resolve()
candidates = (start, *start.parents)
PROJECT_ROOT = next(
    (p for p in candidates if (p / "pyproject.toml").is_file()
     and (p / "src" / "phishguard_vn").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Chuyen working directory vao repo hoac dat PHISHGUARD_ROOT "
        "toi thu muc du an tren may ban."
    )
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from phishguard_vn.paths import get_data_dir, get_config_path
DATA_DIR = get_data_dir()
with get_config_path().open("rb") as handle:
    CONFIG = tomllib.load(handle)

print("Project:", PROJECT_ROOT)
print("Dataset:", DATA_DIR)
print("Config:", CONFIG["experiment"]["name"])


Project: D:\Intro_ML\PhishGuardVN
Dataset: D:\Intro_ML\PhishGuardVN\data
Config: week4_e1_core_lr


## 1. Nạp và kiểm kê
Đọc tài liệu gốc; kiểm tra cột bắt buộc, kiểu dữ liệu, số dòng bảng chính và các split. Không nối dọc những bảng có cùng URL.

In [37]:
import pandas as pd
from IPython.display import display

from phishguard_vn.data import (
    load_dataset,
    audit_quality,
    prepare_core,
)

# Load the original dataset and audit its quality.
dataset_path = DATA_DIR / CONFIG["data"]["dataset_file"]
frame = load_dataset(dataset_path)
quality = audit_quality(frame)

print(f"Dataset path: {dataset_path}")
print(f"Number of rows: {frame.shape[0]:,}")
print(f"Number of columns: {frame.shape[1]}")

display(frame.head())

# Inspect column types, missing values and distinct values.
inventory = pd.DataFrame({
    "data_type": frame.dtypes.astype(str),
    "missing_count": frame.isna().sum(),
    "missing_pct": (frame.isna().mean() * 100).round(2),
    "unique_values": frame.nunique(dropna=False),
})

display(inventory)

D:\Intro_ML\PhishGuardVN\src\phishguard_vn\data.py:19: DtypeWarning: Columns (6: impersonated_org, 21: status) have mixed types. Specify dtype option on import or set low_memory=False.
  frame = pd.read_csv(


Dataset path: D:\Intro_ML\PhishGuardVN\data\dataset_url.csv
Number of rows: 53,116
Number of columns: 29


,channel,label,source,is_llm,lang,scenario,impersonated_org,collected_at,scraped_at,label_source,...,has_at,suspicious_tld,status,tier,rank,text,msg_len,gen_model,has_url,id
0,url,phishing,tinnhiemmang,0,vi,ecommerce,NaN,18/02/2025,2026-07-13T08:50:44.873070+00:00,feed,...,0.0,0.0,Đang xử lý,silver,NaN,NaN,NaN,NaN,NaN,75a477a7-aed5-5906-8d6c-e1fc739ef9fd
1,url,phishing,tinnhiemmang,0,vi,telecom,NaN,18/02/2025,2026-07-13T08:50:44.873070+00:00,feed,...,0.0,1.0,Đang xử lý,silver,NaN,NaN,NaN,NaN,NaN,939ebc8f-c54d-50f2-84e7-f02ed5a6aca5
2,url,phishing,tinnhiemmang,0,vi,gov,NaN,18/02/2025,2026-07-13T08:50:44.873070+00:00,feed,...,0.0,0.0,Đang xử lý,silver,NaN,NaN,NaN,NaN,NaN,e3042cd5-fbb9-592c-a8af-0f9c61197b4d
3,url,phishing,tinnhiemmang,0,vi,other,NaN,18/02/2025,2026-07-13T08:50:44.873070+00:00,feed,...,0.0,1.0,Đang xử lý,silver,NaN,NaN,NaN,NaN,NaN,bb00d3eb-a135-5e3c-8b17-e0740198c8f4
4,url,phishing,tinnhiemmang,0,vi,other,NaN,18/02/2025,2026-07-13T08:50:44.873070+00:00,feed,...,0.0,1.0,Đang xử lý,silver,NaN,NaN,NaN,NaN,NaN,dd9e7f0a-bf91-541c-b6fd-20b8a4b7d012


,data_type,missing_count,missing_pct,unique_values
channel,str,0,0.00,2
label,string,0,0.00,2
source,str,0,0.00,7
is_llm,int64,0,0.00,1
lang,str,0,0.00,2
scenario,str,0,0.00,9
impersonated_org,str,51090,96.19,2025
collected_at,str,31455,59.22,954
scraped_at,str,0,0.00,1362
label_source,str,0,0.00,2


## 2. Missing, trùng, xung đột và bất thường
Thống kê missing/constant, URL/ID trùng, nhãn xung đột, URL lỗi. Đối chiếu url_norm thực tế với mô tả. Không loại hàng chỉ vì metadata trống.

In [38]:
# 1. Missing values
missing_table = pd.DataFrame({
    "missing_count": pd.Series(quality["missing_by_column"]),
})
missing_table["missing_pct"] = (
    missing_table["missing_count"] / len(frame) * 100
).round(2)

missing_table = missing_table[
    missing_table["missing_count"] > 0
].sort_values("missing_count", ascending=False)

display(missing_table)

# 2. Constant columns
print("Constant columns:")
print(quality["constant_columns"])

# 3. Duplicates
duplicates_table = pd.json_normalize(
    quality["duplicates"], sep="."
).T

duplicates_table.columns = ["count"]
display(duplicates_table)

# 4. Label conflicts
print("Label conflicts:")
print(quality["label_conflicts"])
# 5.  URL and url_norm
raw_url = frame["url"].astype("string")
norm_url = frame["url_norm"].astype("string")
raw_clean = raw_url.str.strip()

raw_nonblank = (
    raw_url.notna()
    & raw_clean.ne("").fillna(False)
)
norm_nonblank = (
    norm_url.notna()
    & norm_url.str.strip().ne("").fillna(False)
)

url_audit = pd.DataFrame({
    "id": frame["id"],
    "url": raw_url,
    "url_norm": norm_url,
    "missing_url": raw_url.isna(),
    "blank_url": raw_clean.eq("").fillna(False),
    "outer_whitespace": (
        raw_url.notna()
        & raw_url.ne(raw_clean).fillna(False)
    ),
    "missing_url_norm": norm_url.isna(),
    "blank_url_norm": norm_url.str.strip().eq("").fillna(False),
    "outer_whitespace_norm": (
        norm_url.notna()
        & norm_url.ne(norm_url.str.strip()).fillna(False)
    ),
})

url_issue_columns = [
    "missing_url",
    "blank_url",
    "outer_whitespace",
    "missing_url_norm",
    "blank_url_norm",
    "outer_whitespace_norm",
]

url_issue_mask = url_audit[url_issue_columns].any(axis=1)

print("\nRows with URL quality issues:", int(url_issue_mask.sum()))
display(url_audit.loc[url_issue_mask].head(20))

# Check duplicate non-empty raw and normalized URLs
valid_raw = raw_url[raw_nonblank]
valid_norm = norm_url[norm_nonblank]

print(
    "Duplicate non-empty raw URLs after first:",
    int(valid_raw.duplicated().sum()),
)
print(
    "Duplicate non-empty url_norm values after first:",
    int(valid_norm.duplicated().sum()),
)

# 6. Check whether url_norm follows the basic normalization rule:
# strip outer whitespace and add http:// if no HTTP(S) scheme exists.
expected_norm = raw_clean.str.lower()

has_scheme = expected_norm.str.match(
    r"^https?://"
).fillna(False)

needs_scheme = raw_nonblank & ~has_scheme

expected_norm.loc[needs_scheme] = (
    "http://" + expected_norm.loc[needs_scheme]
)

comparable = raw_nonblank & norm_nonblank

matches_rule = (
    comparable
    & expected_norm.eq(norm_url).fillna(False)
)

mismatches_rule = (
    comparable
    & expected_norm.ne(norm_url).fillna(True)
)

print("\nURL normalization audit:")
print("Comparable rows:", int(comparable.sum()))
print("Rows matching the normalization rule:", int(matches_rule.sum()))
print("Rows not matching the normalization rule:", int(mismatches_rule.sum()))

display(
    pd.DataFrame({
        "id": frame.loc[mismatches_rule, "id"],
        "url": raw_url.loc[mismatches_rule],
        "expected_url_norm": expected_norm.loc[mismatches_rule],
        "actual_url_norm": norm_url.loc[mismatches_rule],
    }).head(20)
)

,missing_count,missing_pct
has_url,53116,100.00
gen_model,53116,100.00
msg_len,53116,100.00
text,53116,100.00
impersonated_org,51090,96.19
status,50529,95.13
rank,44611,83.99
collected_at,31455,59.22


Constant columns:
['is_llm', 'has_ip', 'text', 'msg_len', 'gen_model', 'has_url']


,count
exact_duplicate_rows,0
id.repeated rows after first,0
id.unique duplicate values,0
url.repeated rows after first,0
url.unique duplicate values,0


Label conflicts:
{'by_id': {'group count': 0, 'examples': []}, 'by_url': {'group count': 0, 'examples': []}}

Rows with URL quality issues: 0


,id,url,url_norm,missing_url,blank_url,outer_whitespace,missing_url_norm,blank_url_norm,outer_whitespace_norm


Duplicate non-empty raw URLs after first: 0
Duplicate non-empty url_norm values after first: 0

URL normalization audit:
Comparable rows: 53116
Rows matching the normalization rule: 53116
Rows not matching the normalization rule: 0


,id,url,expected_url_norm,actual_url_norm


## 3. Chuẩn bị core và bảng bàn giao
Lọc gold/silver; giữ raw URL, id, label, tier, source và split. Ghi số mẫu trước/sau xử lý; giữ bronze riêng.

In [39]:
# Select gold/silver as the core dataset.
core = prepare_core(frame)

# Keep bronze separate.
bronze = frame.loc[frame["tier"].eq("bronze")].copy()

# Record rows excluded from the core because of their tier.
excluded = frame.loc[~frame.index.isin(core.index), ["id", "tier"]].copy()
excluded["reason"] = "tier_not_in_core_gold_silver"

print(f"Original rows: {len(frame):,}")
print(f"Core rows: {len(core):,}")
print(f"Bronze rows kept separately: {len(bronze):,}")
print(f"Rows excluded from core: {len(excluded):,}")

print("Core distribution by tier, split and label:")
display(
    core.groupby(["tier", "split", "label"])
    .size()
    .unstack("label", fill_value=0)
)

print("ID unique:", core["id"].is_unique)
print("ID non-missing:", core["id"].notna().all())

Original rows: 53,116
Core rows: 18,997
Bronze rows kept separately: 34,119
Rows excluded from core: 34,119
Core distribution by tier, split and label:


label         benign  phishing
tier   split                  
gold   test     2604        69
       train    5021      1619
       val       280         0
silver test     1241       418
       train    5983       457
       val      1281        24

ID unique: True
ID non-missing: True


In [40]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

core.to_csv(OUTPUT_DIR / "core_gold_silver.csv", index=False)
bronze.to_csv(OUTPUT_DIR / "bronze_separate.csv", index=False)
excluded.to_csv(OUTPUT_DIR / "core_exclusions.csv", index=False)

print(f"Intermediate tables saved to: {OUTPUT_DIR}")

Intermediate tables saved to: D:\Intro_ML\PhishGuardVN\data\interim


## 4. Ý nghĩa nhãn và abuse_type
Nối theo url, kiểm tra bao phủ lớp dương, phân bố abuse_type và unknown. Diễn giải giới hạn nhãn, không dùng abuse_type làm X.

In [41]:
abuse_path = DATA_DIR / "abuse_type.csv"
abuse = pd.read_csv(abuse_path, low_memory=False)

print("Columns in abuse_type.csv:")
print(abuse.columns.tolist())

print("Shape:", abuse.shape)
display(abuse.head())

Columns in abuse_type.csv:
['url', 'abuse_type', 'rule']
Shape: (36706, 3)


,url,abuse_type,rule
0,tiktokshopseller-vn.com,brand_impersonation,scenario=ecommerce
1,vtp-vandon.online,brand_impersonation,scenario=telecom
2,chinhphu2025.com,brand_impersonation,scenario=gov
3,dvc.bcavn.cc,unknown,NaN
4,8965.crnikjhek.top,unknown,NaN


In [ ]:
phishing = frame.loc[
    frame["label"].eq("phishing"),
    ["id", "url", "label", "tier", "split"],
].copy()

abuse_for_join = abuse[["url", "abuse_type", "rule"]].copy()


phishing["url"] = phishing["url"].astype("string")
abuse_for_join["url"] = abuse_for_join["url"].astype("string")

print("Phishing samples:", len(phishing))
print("Abuse annotation rows:", len(abuse_for_join))
print(
    "Duplicate URLs in phishing samples:",
    int(phishing["url"].duplicated().sum()),
)
print(
    "Duplicate URLs in abuse annotations:",
    int(abuse_for_join["url"].duplicated().sum()),
)


phishing_abuse = phishing.merge(
    abuse_for_join,
    on="url",
    how="left",
    indicator=True,
    validate="one_to_one",
)

matched = phishing_abuse.loc[
    phishing_abuse["_merge"].eq("both")
].copy()

unmatched = phishing_abuse.loc[
    phishing_abuse["_merge"].eq("left_only"),
    ["id", "url", "tier", "split"],
].copy()

coverage = len(matched) / len(phishing) * 100 if len(phishing) else 0

print(f"\nMatched phishing samples: {len(matched):,}")
print(f"Unmatched phishing samples: {len(unmatched):,}")
print(f"Annotation coverage: {coverage:.2f}%")

display(unmatched.head(20))

type_counts = (
    matched["abuse_type"]
    .value_counts(dropna=False)
    .rename_axis("abuse_type")
    .reset_index(name="count")
)

if len(matched):
    type_counts["percentage"] = (
        type_counts["count"] / len(matched) * 100
    ).round(2)
else:
    type_counts["percentage"] = 0.0

display(type_counts)

subtype_by_split = (
    matched.groupby(["tier", "split", "abuse_type"], dropna=False)
    .size()
    .unstack("abuse_type", fill_value=0)
)

display(subtype_by_split)

benign = frame.loc[
    frame["label"].eq("benign"),
    ["id", "url", "tier", "split"],
].copy()

benign["url"] = benign["url"].astype("string")

benign_matches = benign.merge(
    abuse_for_join,
    on="url",
    how="inner",
    validate="one_to_one",
)

print("Benign samples matched to abuse annotations:", len(benign_matches))
display(benign_matches.head(20))

Phishing samples: 36706
Abuse annotation rows: 36706
Duplicate URLs in phishing samples: 0
Duplicate URLs in abuse annotations: 0

Matched phishing samples: 36,706
Unmatched phishing samples: 0
Annotation coverage: 100.00%


,id,url,tier,split


,abuse_type,count,percentage
0,unknown,32666,88.99
1,brand_impersonation,3364,9.16
2,gambling,337,0.92
3,investment,284,0.77
4,betting_stream,40,0.11
5,adult,15,0.04


abuse_type    adult  betting_stream  brand_impersonation  gambling  \
tier   split                                                         
bronze test       2               0                  673         4   
       train     11              18                 1798       319   
       val        0               0                  442        10   
gold   test       0               0                   35         0   
       train      2              15                  176         1   
silver test       0               1                  110         3   
       train      0               6                  127         0   
       val        0               0                    3         0   

abuse_type    investment  unknown  
tier   split                       
bronze test           57     3873  
       train         164    21519  
       val            44     5185  
gold   test            1       33  
       train          16     1409  
silver test            1      303  
       train           1      323  
       val             0       21

Benign samples matched to abuse annotations: 0


,id,url,tier,split,abuse_type,rule


### Ý nghĩa nhãn và độ bao phủ của dữ liệu abuse_type

Trong bộ dữ liệu chính có 36.706 mẫu phishing. Khi nối với bảng `abuse_type.csv` bằng URL nguyên bản, cả 36.706 mẫu đều tìm được bản ghi tương ứng, đạt độ bao phủ 100%. Không phát hiện URL trùng lặp trong cả hai bảng và không có mẫu benign nào khớp với bảng chú giải loại hình tấn công.

Loại `unknown` chiếm 32.666 mẫu (88,99%), là nhóm chiếm tỷ lệ lớn nhất. Trong các loại tấn công đã được xác định cụ thể, `brand_impersonation` phổ biến nhất với 3.364 mẫu (9,16%). Các nhóm còn lại gồm `gambling` với 337 mẫu, `investment` với 284 mẫu, `betting_stream` với 40 mẫu và `adult` với 15 mẫu.

Tỷ lệ `unknown` cao (88,99%) hạn chế khả năng phân tích sâu theo từng loại hình phishing cụ thể.

Hai cột `abuse_type` và `rule` được sử dụng để phân tích chú giải và kiểm tra chất lượng dữ liệu, không được đưa vào làm đặc trưng đầu vào của mô hình baseline, nhằm tránh rò rỉ thông tin liên quan đến nhãn.

## 5. Kết luận và bàn giao


## 5. Kết luận và bàn giao

### Kết quả chính

Bộ dữ liệu gốc gồm 53.116 dòng và 29 cột. Các phép kiểm tra chưa phát hiện bản ghi trùng hoàn toàn, ID/URL trùng theo chuỗi nguyên bản hoặc xung đột nhãn. Một số cột metadata có tỷ lệ thiếu cao; chi tiết đã được trình bày ở mục 2. Dữ liệu gốc được giữ nguyên.

### Chuẩn bị và bàn giao dữ liệu

Tập core gồm 18.997 bản ghi thuộc tier `gold` và `silver`; 34.119 bản ghi `bronze` được giữ riêng. Quá trình chuẩn bị bảo toàn `id`, `url`, `label`, `tier` và `split`. Các bảng được nối theo khóa phù hợp, không ghép theo vị trí dòng.

Các bảng đầu ra dự kiến gồm `core_gold_silver.csv`, `bronze_separate.csv` và `core_exclusions.csv` trong `data/interim/`. Chỉ ghi nhận các file này là đầu ra bàn giao sau khi đã tạo và kiểm tra thành công.

### Vấn đề cần phối hợp

Tập validation core chỉ có 24 mẫu phishing và tập Gold validation không có mẫu phishing. Hưng và Phúc cần lưu ý hạn chế này khi thiết kế đánh giá và so sánh baseline. Nhóm cần thống nhất phương án đánh giá trước khi thay đổi split chính thức.

Kiểm tra `url_norm` hiện khớp với quy tắc chuẩn hóa đang được triển khai trong notebook. Cần bảo đảm quy tắc này phù hợp với định nghĩa trong `docs/schema.md` trước khi xác nhận hoàn toàn tính nhất quán của dữ liệu.